In [1]:
import sys
import json
import pandas as pd
import numpy as np
from deltalake import DeltaTable, write_deltalake

In [2]:
sys.path.append(str("/root/capsule/src/"))

In [3]:
from connects_common_connectivity.models import Cluster, ClusterMembership
from connects_common_connectivity.arrow_utils import build_arrow_schema, models_to_table, attach_linkml_metadata

# Tasic 2018 T-types

In [4]:
tasic_2018_anno_file = "../data/visp-patchseq-taxonomy-info/anno.feather"

In [5]:
ref_anno_df = pd.read_feather(tasic_2018_anno_file)

In [6]:
# Color information
cluster_colors = dict(zip(ref_anno_df.cluster_label, ref_anno_df.cluster_color))
subclass_colors = dict(zip(ref_anno_df.subclass_label, ref_anno_df.subclass_color))
class_colors = dict(zip(ref_anno_df.class_label, ref_anno_df.class_color))

In [7]:
# Parent information
cluster_to_subclass = dict(zip(ref_anno_df.cluster_label, ref_anno_df.subclass_label))
subclass_to_class = dict(zip(ref_anno_df.subclass_label, ref_anno_df.class_label))

# Child information
children_dict = {}
for c in class_colors.keys():
    children_dict[c] = []
    for sc, parent_c in subclass_to_class.items():
        if parent_c == c:
            children_dict[c].append(sc)

for sc in subclass_colors.keys():
    children_dict[sc] = []
    for cl, parent_sc in cluster_to_subclass.items():
        if parent_sc == sc:
            children_dict[sc].append(cl)

In [8]:
project_id = "tasic_2018_visp_scrnaseq"

In [9]:
taxon_list = []

taxon_list.append(Cluster(
    id="cell",
    children=list(class_colors.keys()),
    level=0,
    hex_color="#000000",
    heirachy_category="major_class", # sic regarding key name
    project_id=project_id,
))

for class_id, class_color in class_colors.items():
    taxon_list.append(Cluster(
        id=class_id,
        parent="cell",
        children=children_dict[class_id],
        level=1,
        hex_color=class_color,
        heirachy_category="class", # sic regarding key name
        project_id=project_id,
    ))

for subclass_id, subclass_color in subclass_colors.items():
    taxon_list.append(Cluster(
        id=subclass_id,
        parent=subclass_to_class[subclass_id],
        children=children_dict[subclass_id],
        level=2,
        hex_color=subclass_color,
        heirachy_category="subclass", # sic regarding key name
        project_id=project_id,
    ))

for cluster_id, cluster_color in cluster_colors.items():
    taxon_list.append(Cluster(
        id=cluster_id,
        parent=cluster_to_subclass[cluster_id],
        level=3,
        hex_color=cluster_color,
        heirachy_category="cluster", # sic regarding key name
        project_id=project_id,
    ))

In [10]:
schema = build_arrow_schema(Cluster)
table = models_to_table(taxon_list, schema)
table = attach_linkml_metadata(table, linkml_class="Cluster")  # version auto-populated

In [11]:
PATH = "../results/cluster/"
write_deltalake(PATH, table, mode="append", partition_by=["project_id"])

# VISp MET-types

In [12]:
with open("../data/visp-patchseq-taxonomy-info/met_type_colors.json", "r") as f:
    met_colors = json.load(f)

In [13]:
gaba_met_types = [t for t in list(met_colors.keys()) if "MET" in t]
glut_met_types = [t for t in list(met_colors.keys()) if "MET" not in t]

In [14]:
taxon_list = []

taxon_list.append(Cluster(
    id="cell",
    children=["GABAergic", "Glutamatergic"],
    level=0,
    hex_color="#000000",
    heirachy_category="major_class", # sic regarding key name
    project_id="visp_met_types",
))

taxon_list.append(Cluster(
    id="GABAergic",
    parent="cell",
    children=gaba_met_types,
    level=1,
    hex_color=class_colors["GABAergic"],
    heirachy_category="class", # sic regarding key name
    project_id="visp_met_types",
))

taxon_list.append(Cluster(
    id="Glutamatergic",
    parent="cell",
    children=glut_met_types,
    level=1,
    hex_color=class_colors["Glutamatergic"],
    heirachy_category="class", # sic regarding key name
    project_id="visp_met_types",
))

for t in gaba_met_types:
    taxon_list.append(Cluster(
        id=t,
        parent="GABAergic",
        level=2,
        hex_color=met_colors[t],
        heirachy_category="cluster", # sic regarding key name
        project_id="visp_met_types",
    ))

for t in glut_met_types:
    taxon_list.append(Cluster(
        id=t,
        parent="Glutamatergic",
        level=2,
        hex_color=met_colors[t],
        heirachy_category="cluster", # sic regarding key name
        project_id="visp_met_types",
    ))

In [15]:
schema = build_arrow_schema(Cluster)
table = models_to_table(taxon_list, schema)
table = attach_linkml_metadata(table, linkml_class="Cluster")  # version auto-populated


In [16]:
PATH = "../results/cluster/"
write_deltalake(PATH, table, mode="append", partition_by=["project_id"])